In [ ]:
%run generator_benchmark.py
print("✅ Generated perturbation_benchmark.jsonl with 7×500 pairs (3500 total).")

In [ ]:
import json, random
from collections import defaultdict

dataset_file = "perturbation_benchmark.jsonl"
with open(dataset_file, "r") as f:
    data = [json.loads(line) for line in f]

by_slice = defaultdict(list)
for ex in data:
    by_slice[ex["slice"]].append(ex)

for slice_name, examples in by_slice.items():
    print(f"\n=== Slice: {slice_name} ({len(examples)} pairs) ===")
    for ex in random.sample(examples, 3):
        print(f" - Original : {ex['original']}")
        print(f"   Perturbed: {ex['perturbed']}")

In [ ]:
from spotcheck_and_filter_rebalance import filter_and_rebalance

cleaned, variance = filter_and_rebalance(
    in_file="perturbation_benchmark.jsonl",
    out_file="perturbation_benchmark_clean.jsonl"
)

print("\n=== Variance Report (from notebook) ===")
for slice_name, stats in variance.items():
    print(f"{slice_name}: sim={stats['avg_semantic_sim']:.3f}, "
          f"edit={stats['avg_edit_distance']:.2f}")

In [ ]:
import json, random
from collections import defaultdict

with open("perturbation_benchmark_clean.jsonl") as f:
    cleaned = [json.loads(line) for line in f]

by_slice = defaultdict(list)
for ex in cleaned:
    by_slice[ex["slice"]].append(ex)

# Peek 5 random examples per slice
for slice_name, examples in by_slice.items():
    print(f"\n=== Slice: {slice_name} ({len(examples)} pairs) ===")
    for ex in random.sample(examples, 10):
        print(f" - Original : {ex['original']}")
        print(f"   Perturbed: {ex['perturbed']}")


In [ ]:
# === Variance visualization ===
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer, util
import editdistance
from collections import defaultdict
import pandas as pd

# Use clean style
sns.set(style="whitegrid", font_scale=1.2)

model = SentenceTransformer("all-MiniLM-L6-v2")

def compute_distributions(examples):
    originals = [ex["original"] for ex in examples]
    perturbed = [ex["perturbed"] for ex in examples]
    emb1 = model.encode(originals, convert_to_tensor=True, show_progress_bar=False)
    emb2 = model.encode(perturbed, convert_to_tensor=True, show_progress_bar=False)
    sims = util.cos_sim(emb1, emb2).diagonal().tolist()
    edits = [editdistance.eval(o, p) for o, p in zip(originals, perturbed)]
    return sims, edits

# Group cleaned dataset by slice
by_slice = defaultdict(list)
for ex in cleaned:
    by_slice[ex["slice"]].append(ex)

# --- Per-slice histograms ---
for slice_name, examples in by_slice.items():
    sims, edits = compute_distributions(examples)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.histplot(sims, bins=20, kde=True, ax=axes[0], color="#1f77b4")
    axes[0].set_title("Semantic Similarity")
    axes[0].set_xlabel("Cosine similarity")
    sns.histplot(edits, bins=20, ax=axes[1], color="#2ca02c")
    axes[1].set_title("Edit Distance")
    axes[1].set_xlabel("Edit distance")
    fig.suptitle(f"Variance Distributions – {slice_name.capitalize()}", fontsize=14, weight="bold")
    plt.tight_layout()
    plt.show()

# --- Global boxplots ---
all_sims, all_edits = [], []
for slice_name, examples in by_slice.items():
    sims, edits = compute_distributions(examples)
    all_sims.extend([(slice_name, s) for s in sims])
    all_edits.extend([(slice_name, e) for e in edits])

df_sims = pd.DataFrame(all_sims, columns=["Slice", "Similarity"])
df_edits = pd.DataFrame(all_edits, columns=["Slice", "EditDistance"])

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_sims, x="Slice", y="Similarity", palette="Set2")
plt.title("Semantic Similarity Distribution per Slice", fontsize=14, weight="bold")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_edits, x="Slice", y="EditDistance", palette="Set3")
plt.title("Edit Distance Distribution per Slice", fontsize=14, weight="bold")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
